# Base Model - Multiclass Classification

Leaf Classification は、葉の形状特徴量から 99 種の植物を予測する多値分類です。
提出値は各クラスの確率で、評価指標には multiclass log loss を使います。
クラス比率を各 fold で保つ `StratifiedKFold` で検証します。

この baseline は CSV の数値特徴量を使います。`data/images/` の画像は EDA で確認しますが、
学習には使いません。画像モデルとの組み合わせは発展課題です。

## Preparation

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# %pip install -q catboost

In [ ]:
# ====================================================
# Library
# ====================================================
import os
import gc
import warnings

warnings.filterwarnings("ignore")
import random
import subprocess
import numpy as np
import pandas as pd
from pathlib import Path
import pickle

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    log_loss,
    top_k_accuracy_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

import lightgbm as lgb
import xgboost as xgb
from catboost import Pool, CatBoostClassifier
from catboost.utils import get_gpu_device_count

In [ ]:
# ====================================================
# Configurations
# ====================================================
# 実行環境に合わせて変更してください。
# 例: MAIN_PATH = Path("/content/drive/MyDrive/kaggle/leaf-classification")
MAIN_PATH = Path.cwd()

class CFG:
    VER = 1
    AUTHOR = "Jun-Morita"
    COMPETITION = "leaf-classification"
    MAIN_PATH = MAIN_PATH
    DATA_PATH = MAIN_PATH / "data"
    OOF_DATA_PATH = MAIN_PATH / "oof"
    MODEL_DATA_PATH = MAIN_PATH / "models"
    METHOD_LIST = ["lightgbm", "xgboost", "catboost", "linear"]
    METHOD_WEIGHT_DICT = {"lightgbm": 0.3, "xgboost": 0.3, "catboost": 0.3, "linear": 0.1}
    ACCELERATOR = "auto"  # auto / cpu / gpu
    GPU_DEVICE_ID = 0
    GPU_MIN_ROWS = 50_000  # GPU を自動選択する行数の目安
    LIGHTGBM_DEVICE_TYPE = "cpu"  # cuda は CUDA 対応ビルドが必要
    USE_GPU = False
    LIGHTGBM_USE_GPU = False
    XGBOOST_USE_GPU = False
    CATBOOST_USE_GPU = False
    SEED = 42
    N_SEEDS = 1  # 初回実行を軽くする。精度重視なら 5 などに増やす。
    N_SPLIT = 3
    raw_target_col = "species"
    target_col = "species_label"
    id_col = "id"
    metric = "multi_logloss"
    metric_maximize_flag = False
    CLASS_NAMES = []
    N_CLASSES = 0

    num_boost_round = 1000
    early_stopping_round = 50
    verbose = 100

    lgb_params = {
        "objective": "multiclass",
        "metric": "multi_logloss",
        "learning_rate": 0.05,
        "num_leaves": 12,
        "seed": SEED,
        "verbosity": -1,
    }

    xgb_params = {
        "objective": "multi:softprob",
        "eval_metric": "mlogloss",
        "learning_rate": 0.05,
        "max_depth": 4,
        "seed": SEED,
    }

    cat_params = {
        "loss_function": "MultiClass",
        "eval_metric": "MultiClass",
        "learning_rate": 0.05,
        "iterations": num_boost_round,
        "depth": 4,
        "random_seed": SEED,
    }

In [ ]:
# ====================================================
# Accelerator
# ====================================================
def has_accessible_nvidia_gpu():
    try:
        result = subprocess.run(
            ["nvidia-smi", "-L"],
            check=False,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
    except FileNotFoundError:
        return False
    return result.returncode == 0


def configure_accelerator(n_rows):
    accelerator = CFG.ACCELERATOR.lower()
    if accelerator not in {"auto", "cpu", "gpu"}:
        raise ValueError("ACCELERATOR must be one of: auto, cpu, gpu")
    if CFG.LIGHTGBM_DEVICE_TYPE not in {"cpu", "cuda"}:
        raise ValueError("LIGHTGBM_DEVICE_TYPE must be one of: cpu, cuda")

    # 再実行時に以前の GPU 設定を残さない
    for key in ["device", "tree_method"]:
        CFG.xgb_params.pop(key, None)
    for key in ["task_type", "devices"]:
        CFG.cat_params.pop(key, None)
    for key in ["device_type", "gpu_device_id"]:
        CFG.lgb_params.pop(key, None)

    nvidia_gpu_available = has_accessible_nvidia_gpu()
    if accelerator == "gpu" and not nvidia_gpu_available:
        raise RuntimeError(
            "ACCELERATOR='gpu' ですが CUDA 対応 GPU を確認できません。"
            " nvidia-smi と NVIDIA driver を確認してください。"
        )

    auto_gpu = nvidia_gpu_available and n_rows >= CFG.GPU_MIN_ROWS
    use_gpu = accelerator == "gpu" or (accelerator == "auto" and auto_gpu)
    CFG.USE_GPU = use_gpu

    if accelerator == "auto" and nvidia_gpu_available and not auto_gpu:
        print(
            f"GPU is available, but CPU is selected because n_rows={n_rows:,} "
            f"< GPU_MIN_ROWS={CFG.GPU_MIN_ROWS:,}."
        )

    CFG.XGBOOST_USE_GPU = False
    if use_gpu:
        xgb_cuda_enabled = bool(xgb.build_info().get("USE_CUDA", False))
        if xgb_cuda_enabled:
            CFG.xgb_params.update(
                {
                    "device": f"cuda:{CFG.GPU_DEVICE_ID}",
                    "tree_method": "hist",
                }
            )
            CFG.XGBOOST_USE_GPU = True
        else:
            print("XGBoost: CPU fallback (installed package has no CUDA support).")

    CFG.CATBOOST_USE_GPU = False
    if use_gpu:
        try:
            catboost_gpu_count = get_gpu_device_count()
        except Exception as exc:
            catboost_gpu_count = 0
            print(f"CatBoost GPU detection failed: {exc}")
        if catboost_gpu_count > CFG.GPU_DEVICE_ID:
            CFG.cat_params.update(
                {
                    "task_type": "GPU",
                    "devices": str(CFG.GPU_DEVICE_ID),
                }
            )
            CFG.CATBOOST_USE_GPU = True
        else:
            print("CatBoost: CPU fallback (CUDA device is not available to CatBoost).")

    CFG.LIGHTGBM_USE_GPU = False
    if use_gpu and CFG.LIGHTGBM_DEVICE_TYPE == "cuda":
        if not nvidia_gpu_available:
            raise RuntimeError(
                "LIGHTGBM_DEVICE_TYPE='cuda' ですが CUDA 対応 GPU を確認できません。"
            )
        CFG.lgb_params.update(
            {
                "device_type": "cuda",
                "gpu_device_id": CFG.GPU_DEVICE_ID,
            }
        )
        CFG.LIGHTGBM_USE_GPU = True

    if accelerator == "gpu" and not any(
        [
            CFG.XGBOOST_USE_GPU,
            CFG.CATBOOST_USE_GPU,
            CFG.LIGHTGBM_USE_GPU,
        ]
    ):
        raise RuntimeError("GPU を利用できる勾配ブースティングモデルがありません。")

    print(f"ACCELERATOR: {accelerator}")
    print(f"n_rows: {n_rows:,}")
    print(f"LightGBM device: {'cuda' if CFG.LIGHTGBM_USE_GPU else 'cpu'}")
    print(f"XGBoost device: {'cuda' if CFG.XGBOOST_USE_GPU else 'cpu'}")
    print(f"CatBoost device: {'GPU' if CFG.CATBOOST_USE_GPU else 'CPU'}")

In [ ]:
# 実行前チェック
for path in [CFG.DATA_PATH, CFG.OOF_DATA_PATH, CFG.MODEL_DATA_PATH]:
    path.mkdir(parents=True, exist_ok=True)

for file_name in ["train.csv", "test.csv", "sample_submission.csv"]:
    if not (CFG.DATA_PATH / file_name).exists():
        raise FileNotFoundError(
            f"{CFG.DATA_PATH / file_name} が見つかりません。"
            " MAIN_PATH をデータ配置先に合わせて変更してください。"
        )

In [ ]:
# ====================================================
# Seed everything
# ====================================================
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


seed_everything(CFG.SEED)

In [ ]:
# ====================================================
# Timer
# ====================================================
from time import time


class Timer:
    def __init__(self, logger=None, format_str="{:.3f}[s]", prefix=None, suffix=None, sep=" "):
        if prefix:
            format_str = str(prefix) + sep + format_str
        if suffix:
            format_str = format_str + sep + str(suffix)
        self.format_str = format_str
        self.logger = logger
        self.start = None
        self.end = None

    @property
    def duration(self):
        if self.end is None:
            return 0
        return self.end - self.start

    def __enter__(self):
        self.start = time()

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.end = time()
        out_str = self.format_str.format(self.duration)
        if self.logger:
            self.logger.info(out_str)
        else:
            print(out_str)

In [ ]:
def normalize_probabilities(y_pred):
    y_pred = np.asarray(y_pred, dtype=float)
    if y_pred.ndim != 2 or y_pred.shape[1] != CFG.N_CLASSES:
        raise ValueError(
            f"Expected prediction shape (n_rows, {CFG.N_CLASSES}), got {y_pred.shape}"
        )
    y_pred = np.clip(y_pred, 1e-15, None)
    return y_pred / y_pred.sum(axis=1, keepdims=True)


def calc_score(y_true, y_pred):
    y_pred = normalize_probabilities(y_pred)
    return log_loss(y_true, y_pred, labels=np.arange(CFG.N_CLASSES))


def calc_accuracy(y_true, y_pred):
    return accuracy_score(y_true, np.argmax(y_pred, axis=1))


def calc_top3_accuracy(y_true, y_pred):
    return top_k_accuracy_score(
        y_true,
        y_pred,
        k=3,
        labels=np.arange(CFG.N_CLASSES),
    )

## Read Data

In [ ]:
# ====================================================
# Read Data
# ====================================================
# 基本的なデータ
train_df = pd.read_csv(CFG.DATA_PATH / "train.csv")
test_df = pd.read_csv(CFG.DATA_PATH / "test.csv")
submission_df = pd.read_csv(CFG.DATA_PATH / "sample_submission.csv")

label_encoder = LabelEncoder()
train_df[CFG.target_col] = label_encoder.fit_transform(train_df[CFG.raw_target_col])
CFG.CLASS_NAMES = label_encoder.classes_.tolist()
CFG.N_CLASSES = len(CFG.CLASS_NAMES)

submission_class_names = submission_df.columns.drop(CFG.id_col).tolist()
if CFG.CLASS_NAMES != submission_class_names:
    raise ValueError("LabelEncoder のクラス順と sample_submission.csv の列順が一致しません。")

print(f"train: {train_df.shape}")
print(f"test: {test_df.shape}")
print(f"n_classes: {CFG.N_CLASSES}")

configure_accelerator(n_rows=len(train_df))

## EDA

In [ ]:
display(train_df)
display(test_df)

In [ ]:
display(train_df.describe())
display(test_df.describe())

In [ ]:
class_count_df = (
    train_df[CFG.raw_target_col]
    .value_counts()
    .rename_axis(CFG.raw_target_col)
    .reset_index(name="count")
)
display(class_count_df)
display(class_count_df["count"].describe())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(class_count_df["count"], discrete=True, ax=ax)
ax.set_title("Samples per Class")
ax.set_xlabel("Number of samples")
ax.grid(True, axis="y", linestyle="--", alpha=0.5)
fig.tight_layout()
plt.show()

sample_ids = train_df[CFG.id_col].sample(6, random_state=CFG.SEED).tolist()
image_paths = [CFG.DATA_PATH / "images" / f"{sample_id}.jpg" for sample_id in sample_ids]
available_images = [(sample_id, path) for sample_id, path in zip(sample_ids, image_paths) if path.exists()]
if available_images:
    fig, axes = plt.subplots(2, 3, figsize=(9, 6))
    for ax, (sample_id, image_path) in zip(axes.flat, available_images):
        ax.imshow(plt.imread(image_path), cmap="gray")
        species = train_df.loc[train_df[CFG.id_col] == sample_id, CFG.raw_target_col].iloc[0]
        ax.set_title(f"id={sample_id}\n{species}", fontsize=9)
        ax.axis("off")
    for ax in axes.flat[len(available_images):]:
        ax.axis("off")
    fig.tight_layout()
    plt.show()

In [ ]:
display(train_df.isna().sum())
display(test_df.isna().sum())

## Feature Engineering

In [ ]:
def create_raw_features(input_df):
    df = input_df.copy()
    drop_cols = [CFG.id_col, CFG.raw_target_col, CFG.target_col]
    return df.drop(columns=drop_cols, errors="ignore")

In [ ]:
from typing import List


def build_feature(input_df: pd.DataFrame, feature_functions: List) -> pd.DataFrame:
    # 出力するデータフレームを空で用意して
    out_df = pd.DataFrame()

    print("start build features...")

    # 各特徴生成関数ごとで
    for func in feature_functions:
        with Timer(prefix=f"create {func.__name__}"):
            # 特徴量を作成し
            _df = func(input_df)

        # 横方向 (axis=1) にがっちゃんこ (concat) する
        out_df = pd.concat([out_df, _df], axis=1)

    return out_df

In [ ]:
feature_functions = [
    create_raw_features,
]

In [ ]:
feat_train_df = build_feature(input_df=train_df, feature_functions=feature_functions)
feat_test_df = build_feature(input_df=test_df, feature_functions=feature_functions)

train_full_df = pd.concat([train_df[[CFG.target_col]], feat_train_df], axis=1)

features = feat_train_df.columns.tolist()

categorical_features = []
linear_categorical_features = []
for col in categorical_features:
    full_cat = pd.concat([feat_train_df[col], feat_test_df[col]]).astype("category")
    train_full_df[col] = pd.Categorical(train_full_df[col], categories=full_cat.cat.categories)
    feat_train_df[col] = pd.Categorical(feat_train_df[col], categories=full_cat.cat.categories)
    feat_test_df[col] = pd.Categorical(feat_test_df[col], categories=full_cat.cat.categories)

display(features)
display(train_full_df)
display(feat_test_df)

## Validation and Model Training

多値分類では、fold ごとのクラス比率を揃えるため `StratifiedKFold` を使います。
log loss は正解クラスへ十分な確率を割り当てられたかを評価します。

In [ ]:
model_dict = {}


def lightgbm_training(
    x_train: pd.DataFrame,
    y_train: pd.DataFrame,
    x_valid: pd.DataFrame,
    y_valid: pd.DataFrame,
    features: list,
    categorical_features: list,
    seed: int,
):
    params = {**CFG.lgb_params, "seed": seed, "num_class": CFG.N_CLASSES}
    lgb_train = lgb.Dataset(x_train, y_train, categorical_feature=categorical_features)
    lgb_valid = lgb.Dataset(x_valid, y_valid, categorical_feature=categorical_features)
    model = lgb.train(
        params=params,
        train_set=lgb_train,
        num_boost_round=CFG.num_boost_round,
        valid_sets=[lgb_train, lgb_valid],
        callbacks=[
            lgb.early_stopping(stopping_rounds=CFG.early_stopping_round, verbose=CFG.verbose),
            lgb.log_evaluation(CFG.verbose),
        ],
    )
    # Predict validation
    valid_pred = normalize_probabilities(model.predict(x_valid))
    return model, valid_pred


def xgboost_training(
    x_train: pd.DataFrame,
    y_train: pd.DataFrame,
    x_valid: pd.DataFrame,
    y_valid: pd.DataFrame,
    features: list,
    categorical_features: list,
    seed: int,
):
    params = {**CFG.xgb_params, "seed": seed, "num_class": CFG.N_CLASSES}
    xgb_train = xgb.DMatrix(data=x_train, label=y_train, enable_categorical=True)
    xgb_valid = xgb.DMatrix(data=x_valid, label=y_valid, enable_categorical=True)
    model = xgb.train(
        params,
        dtrain=xgb_train,
        num_boost_round=CFG.num_boost_round,
        evals=[(xgb_train, "train"), (xgb_valid, "eval")],
        early_stopping_rounds=CFG.early_stopping_round,
        verbose_eval=CFG.verbose,
    )
    # Predict validation
    valid_pred = normalize_probabilities(
        model.predict(xgb.DMatrix(x_valid, enable_categorical=True))
    )
    return model, valid_pred


def catboost_training(
    x_train: pd.DataFrame,
    y_train: pd.DataFrame,
    x_valid: pd.DataFrame,
    y_valid: pd.DataFrame,
    features: list,
    categorical_features: list,
    seed: int,
):
    params = {**CFG.cat_params, "random_seed": seed, "classes_count": CFG.N_CLASSES}
    cat_train = Pool(data=x_train, label=y_train, cat_features=categorical_features)
    cat_valid = Pool(data=x_valid, label=y_valid, cat_features=categorical_features)
    model = CatBoostClassifier(**params)
    model.fit(
        cat_train,
        eval_set=[cat_valid],
        early_stopping_rounds=CFG.early_stopping_round,
        verbose=CFG.verbose,
        use_best_model=True,
    )
    # Predict validation
    valid_pred = normalize_probabilities(model.predict_proba(x_valid))
    return model, valid_pred


def linear_training(
    x_train: pd.DataFrame,
    y_train: pd.DataFrame,
    x_valid: pd.DataFrame,
    y_valid: pd.DataFrame,
    features: list,
    categorical_features: list,
    seed: int,
):
    numeric_features = [col for col in features if col not in categorical_features]
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_features,
            ),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                categorical_features,
            ),
        ]
    )
    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", LogisticRegression(max_iter=2000, random_state=seed)),
        ]
    )
    model.fit(x_train, y_train)
    valid_pred = normalize_probabilities(model.predict_proba(x_valid))

    return model, valid_pred


def gradient_boosting_model_cv_training(
    method: str,
    train_df: pd.DataFrame,
    features: list,
    categorical_features: list,
    linear_categorical_features: list,
    seed: int,
):
    # Create a numpy array to store out of folds predictions
    oof_predictions = np.zeros((len(train_df), CFG.N_CLASSES))
    oof_fold = np.zeros(len(train_df))
    target_col = CFG.target_col
    seed_everything(seed)

    # kfoldはタスクによって変更
    kfold = StratifiedKFold(n_splits=CFG.N_SPLIT, shuffle=True, random_state=seed)

    for fold, (train_index, valid_index) in enumerate(
        kfold.split(train_df[features], train_df[CFG.target_col])
    ):
        print("-" * 50)
        print(f"{method} (seed {seed}) training fold {fold+1}")
        x_train = train_df[features].iloc[train_index]
        y_train = train_df[target_col].iloc[train_index]
        x_valid = train_df[features].iloc[valid_index]
        y_valid = train_df[target_col].iloc[valid_index]
        if method == "lightgbm":
            model, valid_pred = lightgbm_training(
                x_train, y_train, x_valid, y_valid, features, categorical_features, seed
            )
            importance_df = pd.DataFrame(
                model.feature_importance(), index=features, columns=["importance"]
            ).reset_index()
            importance_df.to_csv(
                CFG.MODEL_DATA_PATH
                / f"{CFG.AUTHOR}_{method}_{target_col}_fold{fold + 1}_seed{seed}_ver{CFG.VER}_importance.csv",
                index=False,
            )
        if method == "xgboost":
            model, valid_pred = xgboost_training(
                x_train, y_train, x_valid, y_valid, features, categorical_features, seed
            )
        if method == "catboost":
            model, valid_pred = catboost_training(
                x_train, y_train, x_valid, y_valid, features, categorical_features, seed
            )
        if method == "linear":
            model, valid_pred = linear_training(
                x_train,
                y_train,
                x_valid,
                y_valid,
                features,
                linear_categorical_features,
                seed,
            )
        # Save best model
        with open(
            CFG.MODEL_DATA_PATH
            / f"{CFG.AUTHOR}_{method}_{target_col}_fold{fold + 1}_seed{seed}_ver{CFG.VER}.pkl",
            "wb",
        ) as f:
            pickle.dump(model, f)
        model_dict[f"{method}_{target_col}_fold{fold+1}_seed{seed}"] = model

        # Add to out of folds array
        oof_predictions[valid_index] = valid_pred
        del x_train, x_valid, y_train, y_valid, model, valid_pred
        gc.collect()
        oof_fold[valid_index] = fold + 1

    oof_df = pd.DataFrame(oof_predictions, columns=CFG.CLASS_NAMES)
    oof_df["fold"] = oof_fold.astype(int)
    oof_df["row_id"] = train_df.index

    y_true = train_df[CFG.target_col].values
    score = calc_score(y_true, oof_predictions)
    accuracy = calc_accuracy(y_true, oof_predictions)
    top3_accuracy = calc_top3_accuracy(y_true, oof_predictions)
    print(f"{method} (seed {seed}) our out of folds CV {CFG.metric} is {score}")
    print(f"{method} (seed {seed}) accuracy is {accuracy}")
    print(f"{method} (seed {seed}) top-3 accuracy is {top3_accuracy}")

    oof_path = CFG.OOF_DATA_PATH / f"oof_{CFG.AUTHOR}_{method}_seed{seed}_ver{CFG.VER}.csv"
    oof_df.to_csv(oof_path, index=False)

    return score

In [ ]:
for method in CFG.METHOD_LIST:
    scores = []
    for seed_idx in range(CFG.N_SEEDS):
        seed = CFG.SEED + seed_idx
        print("=" * 50)
        print(f"Training {method} with seed {seed} ({seed_idx + 1}/{CFG.N_SEEDS})")
        print("=" * 50)
        score = gradient_boosting_model_cv_training(
            method,
            train_full_df,
            features,
            categorical_features,
            linear_categorical_features,
            seed,
        )
        scores.append(score)

    print("=" * 50)
    print(f"{method} Seed Averaging Results:")
    print(f"  Seeds: {[CFG.SEED + i for i in range(CFG.N_SEEDS)]}")
    print(f"  Scores: {scores}")
    print(f"  Mean: {np.mean(scores):.6f}")
    print(f"  Std: {np.std(scores):.6f}")
    print("=" * 50)

In [ ]:
def load_model(method: str, target_col: str, fold: int, seed: int):
    with open(
        CFG.MODEL_DATA_PATH
        / f"{CFG.AUTHOR}_{method}_{target_col}_fold{fold + 1}_seed{seed}_ver{CFG.VER}.pkl",
        "rb",
    ) as f:
        return pickle.load(f)


def predict_one_model(method: str, model, x_test: pd.DataFrame):
    if method == "xgboost":
        pred = model.predict(xgb.DMatrix(x_test, enable_categorical=True))
    elif method in ["catboost", "linear"]:
        pred = model.predict_proba(x_test)
    else:
        pred = model.predict(x_test)
    return normalize_probabilities(pred)


def gradient_boosting_model_inference(
    method: str, target_col: str, test_df: pd.DataFrame, features: list
):
    x_test = test_df[features]
    predictions = []
    for seed_idx in range(CFG.N_SEEDS):
        seed = CFG.SEED + seed_idx
        for fold in range(CFG.N_SPLIT):
            model = load_model(method, target_col, fold, seed)
            predictions.append(predict_one_model(method, model, x_test))
    return normalize_probabilities(np.mean(predictions, axis=0))


def predicting(input_df: pd.DataFrame, features: list):
    target_col = CFG.target_col
    print(f"{target_col} inference")
    print(
        f"Using {CFG.N_SEEDS} seeds x {CFG.N_SPLIT} folds "
        f"= {CFG.N_SEEDS * CFG.N_SPLIT} models per method"
    )
    blended_pred = np.zeros((len(input_df), CFG.N_CLASSES))
    weight_sum = sum(CFG.METHOD_WEIGHT_DICT.values())
    for method in CFG.METHOD_LIST:
        method_pred = gradient_boosting_model_inference(
            method, target_col, input_df, features
        )
        blended_pred += method_pred * CFG.METHOD_WEIGHT_DICT[method] / weight_sum
    return normalize_probabilities(blended_pred)

## OOF Evaluation

In [ ]:
def load_oof_preds(method: str, train_df: pd.DataFrame) -> np.ndarray:
    # OOFファイル群を読み込み、train_df.index に沿った平均予測を返す
    per_seed_preds = []
    for seed_idx in range(CFG.N_SEEDS):
        seed = CFG.SEED + seed_idx
        path = CFG.OOF_DATA_PATH / f"oof_{CFG.AUTHOR}_{method}_seed{seed}_ver{CFG.VER}.csv"
        oof = pd.read_csv(path)
        if "row_id" not in oof.columns:
            raise ValueError(
                f"OOF file for {method} has no 'row_id'. 学習側で row_id を保存してください。"
            )
        ordered = oof.set_index("row_id").loc[train_df.index]
        per_seed_preds.append(ordered[CFG.CLASS_NAMES].astype(float).values)

    if not per_seed_preds:
        raise ValueError(f"No OOF files found for {method}")

    return normalize_probabilities(np.mean(per_seed_preds, axis=0))


def eval_single_oof_from_file(method: str, train_df: pd.DataFrame) -> dict:
    # 単一メソッドの OOF を読み込んで指標を計算
    y_true = train_df[CFG.target_col].values
    y_pred = load_oof_preds(method, train_df)
    metrics = {
        "method": method,
        CFG.metric: calc_score(y_true, y_pred),
        "accuracy": calc_accuracy(y_true, y_pred),
        "top3_accuracy": calc_top3_accuracy(y_true, y_pred),
    }
    print(f"[{method}] {metrics}")
    return metrics


def eval_blend_oof_from_files(train_df: pd.DataFrame, method_list: list) -> tuple:
    # 複数メソッドの OOF を重み付きブレンドして指標を計算
    weights = np.array([CFG.METHOD_WEIGHT_DICT[m] for m in method_list], dtype=float)
    weights = weights / weights.sum()
    pred_matrix = np.stack([load_oof_preds(m, train_df) for m in method_list])
    blended_pred = normalize_probabilities(
        (pred_matrix * weights[:, np.newaxis, np.newaxis]).sum(axis=0)
    )

    y_true = train_df[CFG.target_col].values
    metrics = {
        "method": f"blend({','.join(method_list)})",
        CFG.metric: calc_score(y_true, blended_pred),
        "accuracy": calc_accuracy(y_true, blended_pred),
        "top3_accuracy": calc_top3_accuracy(y_true, blended_pred),
    }
    print(f"[BLEND] {metrics}")

    return metrics, blended_pred

In [ ]:
metric_rows = []
for m in CFG.METHOD_LIST:
    metric_rows.append(eval_single_oof_from_file(m, train_full_df))

blend_metrics, oof_pred = eval_blend_oof_from_files(train_full_df, CFG.METHOD_LIST)
metric_rows.append(blend_metrics)

metric_df = pd.DataFrame(metric_rows).sort_values(CFG.metric)
display(metric_df)

## Inference and Submission

In [ ]:
test_pred = predicting(feat_test_df, features)

In [ ]:
submission_df[CFG.CLASS_NAMES] = test_pred
if not np.allclose(submission_df[CFG.CLASS_NAMES].sum(axis=1), 1.0):
    raise ValueError("提出用のクラス確率の合計が 1 ではありません。")
submission_df.to_csv(CFG.OOF_DATA_PATH / "submission.csv", index=False)
submission_df

## Feature Importance

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


# 全シードの平均importanceを計算
all_importances = []
for seed_idx in range(CFG.N_SEEDS):
    seed = CFG.SEED + seed_idx
    for fold in range(CFG.N_SPLIT):
        model = load_model("lightgbm", CFG.target_col, fold, seed)
        all_importances.append(model.feature_importance())

mean_importance = np.mean(all_importances, axis=0)
std_importance = np.std(all_importances, axis=0)

importance_df = pd.DataFrame({
    "feature": feat_train_df.columns,
    "importance": mean_importance,
    "std": std_importance,
}).sort_values("importance", ascending=False).head(50)

fig, ax = plt.subplots(figsize=(8, max(6, len(importance_df) * 0.25)))
ax.barh(importance_df["feature"], importance_df["importance"], xerr=importance_df["std"])
ax.invert_yaxis()
ax.set_xlabel("Feature Importance")
ax.set_title(f"Feature Importance (Mean over {CFG.N_SEEDS} seeds x {CFG.N_SPLIT} folds)")
ax.grid()
fig.tight_layout()
plt.show()

## Prediction Diagnostics

In [ ]:
def create_confusion_pairs(y_true, y_pred):
    pred_labels = np.argmax(y_pred, axis=1)
    cm = confusion_matrix(y_true, pred_labels, labels=np.arange(CFG.N_CLASSES))
    rows = []
    for true_label in range(CFG.N_CLASSES):
        for pred_label in range(CFG.N_CLASSES):
            count = cm[true_label, pred_label]
            if true_label != pred_label and count > 0:
                rows.append(
                    {
                        "true_class": CFG.CLASS_NAMES[true_label],
                        "predicted_class": CFG.CLASS_NAMES[pred_label],
                        "count": count,
                    }
                )
    columns = ["true_class", "predicted_class", "count"]
    return pd.DataFrame(rows, columns=columns).sort_values("count", ascending=False)


def create_prediction_diagnostics(input_df, y_true, y_pred):
    pred_labels = np.argmax(y_pred, axis=1)
    confidence = np.max(y_pred, axis=1)
    true_probability = y_pred[np.arange(len(y_true)), y_true]
    top2_probability = np.partition(y_pred, -2, axis=1)[:, -2]
    return pd.DataFrame(
        {
            CFG.id_col: input_df[CFG.id_col].values,
            "true_class": np.array(CFG.CLASS_NAMES)[y_true],
            "predicted_class": np.array(CFG.CLASS_NAMES)[pred_labels],
            "is_correct": y_true == pred_labels,
            "confidence": confidence,
            "top1_top2_margin": confidence - top2_probability,
            "true_class_probability": true_probability,
            "sample_log_loss": -np.log(np.clip(true_probability, 1e-15, None)),
        }
    )


def evaluate_multiclass_predictions(input_df, y_true, y_pred):
    y_pred = normalize_probabilities(y_pred)
    pred_labels = np.argmax(y_pred, axis=1)
    print(f"{CFG.metric}: {calc_score(y_true, y_pred):.6f}")
    print(f"accuracy: {calc_accuracy(y_true, y_pred):.6f}")
    print(f"top-3 accuracy: {calc_top3_accuracy(y_true, y_pred):.6f}")

    report_df = pd.DataFrame(
        classification_report(
            y_true,
            pred_labels,
            labels=np.arange(CFG.N_CLASSES),
            target_names=CFG.CLASS_NAMES,
            output_dict=True,
            zero_division=0,
        )
    ).T
    confusion_pair_df = create_confusion_pairs(y_true, y_pred)
    diagnostics_df = create_prediction_diagnostics(input_df, y_true, y_pred)

    print("Recall が低いクラス")
    display(report_df.loc[CFG.CLASS_NAMES].sort_values("recall").head(20))
    print("混同しやすいクラスの組み合わせ")
    display(confusion_pair_df.head(20))
    print("Log loss が大きい予測")
    display(diagnostics_df.sort_values("sample_log_loss", ascending=False).head(20))

    return report_df, confusion_pair_df, diagnostics_df


y_true = train_full_df[CFG.target_col].values
report_df, confusion_pair_df, diagnostics_df = evaluate_multiclass_predictions(
    train_df, y_true, oof_pred
)

In [ ]:
def plot_confusion_matrix_for_error_prone_classes(y_true, y_pred, top_n=20):
    pred_labels = np.argmax(y_pred, axis=1)
    cm = confusion_matrix(y_true, pred_labels, labels=np.arange(CFG.N_CLASSES))
    errors_per_class = cm.sum(axis=1) - np.diag(cm)
    selected_labels = np.argsort(errors_per_class)[::-1][:top_n]
    selected_names = np.array(CFG.CLASS_NAMES)[selected_labels]

    cm_normalized = cm / np.clip(cm.sum(axis=1, keepdims=True), 1, None)
    selected_cm = cm_normalized[np.ix_(selected_labels, selected_labels)]

    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(
        selected_cm,
        xticklabels=selected_names,
        yticklabels=selected_names,
        cmap="Blues",
        vmin=0,
        vmax=1,
        ax=ax,
    )
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title(f"Normalized Confusion Matrix: {top_n} Error-Prone Classes")
    fig.tight_layout()
    plt.show()


plot_confusion_matrix_for_error_prone_classes(y_true, oof_pred, top_n=20)

In [ ]:
def plot_confidence_diagnostics(diagnostics_df):
    plot_df = diagnostics_df.copy()
    plot_df["result"] = plot_df["is_correct"].map({True: "correct", False: "incorrect"})

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.histplot(
        data=plot_df,
        x="confidence",
        hue="result",
        bins=20,
        stat="density",
        common_norm=False,
        alpha=0.5,
        ax=axes[0],
    )
    axes[0].set_title("Confidence Distribution")
    axes[0].grid(True, linestyle="--", alpha=0.5)

    plot_df["confidence_bin"] = pd.cut(
        plot_df["confidence"],
        bins=np.linspace(0, 1, 11),
        include_lowest=True,
    )
    calibration_df = (
        plot_df.groupby("confidence_bin", observed=False)
        .agg(
            samples=("is_correct", "size"),
            mean_confidence=("confidence", "mean"),
            accuracy=("is_correct", "mean"),
        )
        .dropna()
        .reset_index()
    )
    axes[1].plot([0, 1], [0, 1], color="gray", linestyle="--", label="ideal")
    sns.lineplot(
        data=calibration_df,
        x="mean_confidence",
        y="accuracy",
        marker="o",
        label="OOF",
        ax=axes[1],
    )
    axes[1].set_xlim(0, 1)
    axes[1].set_ylim(0, 1)
    axes[1].set_title("Confidence Calibration")
    axes[1].grid(True, linestyle="--", alpha=0.5)
    axes[1].legend()
    fig.tight_layout()
    plt.show()

    return calibration_df


calibration_df = plot_confidence_diagnostics(diagnostics_df)
display(calibration_df)

In [ ]:
def plot_method_confidence_hist(method_list, train_df, bins=30):
    bin_edges = np.linspace(0.0, 1.0, bins + 1)
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for method in method_list:
        pred = load_oof_preds(method, train_df)
        sns.histplot(
            np.max(pred, axis=1),
            label=f"OOF ({method})",
            stat="density",
            bins=bin_edges,
            element="step",
            fill=False,
            ax=ax,
        )
    ax.set_xlim(0, 1)
    ax.set_xlabel("Maximum predicted probability")
    ax.set_ylabel("Density")
    ax.set_title("Confidence Distribution by Method")
    ax.legend()
    ax.grid(True, linestyle="--", alpha=0.7)
    fig.tight_layout()
    plt.show()


plot_method_confidence_hist(CFG.METHOD_LIST, train_full_df, bins=30)